#### This notebook shows how to read the CMRxRecon dataset and apply some simple transformations to the data.
#### We utilized some useful codes from fastMRI repo.

In [1]:
%matplotlib inline
%matplotlib widget

import h5py
import numpy as np
from matplotlib import pyplot as plt
import hdf5storage


The CMRxRecon dataset is distributed as a set of HDF5 files and can be read with the h5py package. Here, we first show how to open a file from the multi-coil dataset. Each file corresponds to ntime * nslice of MRI scan and contains the k-space data.

In [ ]:
file_name = 'data/raw/cmrxrecon/ChallengeData/MultiCoil/Cine/TrainingSet/FullSample/P001/cine_sax.mat'
hf_m = h5py.File(file_name)

In [2]:
def load_and_prepare(file_name):
    hf_m = h5py.File(file_name)
    newvalue = hf_m[list(hf_m.keys())[0]]
    fullmulti = newvalue["real"] + 1j*newvalue["imag"]
    return fullmulti

In [ ]:
# acc4_fullmulti = load_and_prepare('data/raw/cmrxrecon/ChallengeData/MultiCoil/Cine/TrainingSet/AccFactor04/P001/cine_sax.mat')
# acc8_fullmulti = load_and_prepare('data/raw/cmrxrecon/ChallengeData/MultiCoil/Cine/TrainingSet/AccFactor08/P001/cine_sax.mat')
# acc10_fullmulti = load_and_prepare('data/raw/cmrxrecon/ChallengeData/MultiCoil/Cine/TrainingSet/AccFactor10/P001/cine_sax.mat')

In [ ]:
# here to load the kspace data
newvalue = hf_m['kspace_full']
fullmulti = newvalue["real"] + 1j*newvalue["imag"]

In [ ]:
[nframe, nslice, ncoil, ny, nx] = fullmulti.shape
print(nframe, nslice, ncoil, ny, nx)

In multi-coil MRIs, k-space has the following shape: 

#number of frame, number of slices, number of coils, ny, nx (ny is the phase encoding direction for undersampling)

For single-coil MRIs, k-space has the following shape: #number of frame, number of slices, ny, nx


In [ ]:
# choose the first frame and the middle slice
slice_kspace = fullmulti[0,5] 

Let's see what the absolute value of k-space looks like:

In [ ]:
def show_coils(data, slice_nums, cmap=None, vmax = 0.0005):
    fig = plt.figure()
    for i, num in enumerate(slice_nums):
        plt.subplot(1, len(slice_nums), i + 1)
        plt.imshow(data[num], cmap=cmap,vmax=vmax)

In [ ]:
show_coils(np.log(np.abs(slice_kspace) + 1e-9), [0], cmap='gray', vmax = 0.0001)  # This shows coils 0, 5 and 9
# show_coils(np.abs(slice_kspace) - np.abs(acc4_fullmulti[0,5]), [0], cmap='gray', vmax = 0.0001)  # This shows coils 0, 5 and 9
# show_coils(np.log(np.abs(acc4_fullmulti[0,5]) + 1e-9), [0, 4, 9], cmap='gray', vmax = 0.0001)  # This shows coils 0, 5 and 9
# show_coils(np.log(np.abs(acc8_fullmulti[0,5]) + 1e-9), [0, 4, 9], cmap='gray', vmax = 0.0001)  # This shows coils 0, 5 and 9
# show_coils(np.log(np.abs(acc4_fullmulti[0,5]) + 1e-9), [0], cmap='gray', vmax = 0.0001)  # This shows coils 0, 5 and 9

We acknowledged fastMRI repo for their utlity functions to convert k-space into image space. These functions work on PyTorch Tensors. The to_tensor function can convert Numpy arrays to PyTorch Tensors.

In [35]:
import fastmri
from fastmri.data import transforms as T
import torch

In [98]:
patient = 5
ndx = [1,2,7,8,9,10,11]

In [99]:
unders_corr =load_and_prepare(f'data/raw/cmrxrecon/ChallengeData/SingleCoil/Cine/TrainingSet/AccFactor04/P00{patient}/cine_sax.mat')
unders_corr_t = T.to_tensor(unders_corr)
unders_corr_t[ndx] = -unders_corr_t[ndx]
under_imsp = fastmri.ifft2c(unders_corr_t)[:, 0]

In [100]:
test2 = load_and_prepare(f'data/raw/cmrxrecon/ChallengeData/SingleCoil/Cine/TrainingSet/FullSample/P00{patient}/cine_sax.mat')
test2_t = T.to_tensor(test2)
test2_t[ndx] = -test2_t[ndx]
test2_imsp = fastmri.ifft2c(test2_t)[:, 0]

In [ ]:
# torch.save(test2_t, f'data/processed/cmrxrecon/test/training_p00{patient}_single_coil_full_cine_sax.pt')
# torch.save(unders_corr_t, f'data/processed/cmrxrecon/test/training_p00{patient}_single_coil_acc_04_cine_sax.pt')

In [ ]:
f, a= plt.subplots(12,2)
f.set_size_inches(8, 24)
for t in range(12):
    for chan in range(2):
        a[t, chan].imshow(under_imsp[t,...,chan], vmax=0.002, vmin=-0.002, cmap="gray")
        a[t, chan].set_title(f'time {t} channel {chan}')
        a[t, chan].axis("off")
f.tight_layout()

In [ ]:
f, a= plt.subplots(12,2)
f.set_size_inches(8, 24)
for t in range(12):
    for chan in range(2):
        a[t, chan].imshow(test2_imsp[t,...,chan], vmax=0.002, vmin=-0.002, cmap="gray")
        a[t, chan].set_title(f'time {t} channel {chan}')
        a[t, chan].axis("off")
f.tight_layout()

In [ ]:
import torch
t1 = torch.load('data/processed/cmrxrecon/test/training_p001_single_coil_full_cine_sax.pt')
t1.shape

In [ ]:
singcoil_fullmulti = load_and_prepare('data/raw/cmrxrecon/ChallengeData/SingleCoil/Cine/TrainingSet/FullSample/P001/cine_sax.mat')
middle_sing_coil = singcoil_fullmulti[:,0]

In [ ]:
sing_coil_t = T.to_tensor(middle_sing_coil)
sing_coil_image = fastmri.ifft2c(sing_coil_t)
sing_coil_image_abs = fastmri.complex_abs(sing_coil_image)
sing_coil_t.shape, sing_coil_image.shape, sing_coil_image_abs.shape

In [ ]:
singcoil_acc4 = load_and_prepare('data/raw/cmrxrecon/ChallengeData/SingleCoil/Cine/TrainingSet/AccFactor04/P001/cine_sax.mat')[:,0]
singacc4_t = T.to_tensor(singcoil_acc4)
singacc4_image = fastmri.ifft2c(singacc4_t)
singacc4_image_abs = fastmri.complex_abs(singacc4_image)
singacc4_abs = fastmri.complex_abs(singacc4_t)
singacc4_abs.shape, singacc4_image_abs.shape

In [ ]:
singacc4_t.shape

In [ ]:
from src.data.data_utils import FTAndSubsample, ZeroFillAndIFT
subsampled = singacc4_t[singacc4_t != 0].reshape(12, -1, 512, 2)
inmethod = ZeroFillAndIFT((singacc4_t[:1] != 0).movedim(-1,1))
forw = FTAndSubsample((singacc4_t[:1] != 0).movedim(-1,1))
im = inmethod(subsampled[:1].movedim(-1,1))
im_abs = fastmri.complex_abs(im.movedim(1,-1))
forw_inv = forw(im).movedim(1,-1)

In [ ]:
f,a = plt.subplots(1)
a.imshow(im_abs.squeeze(0), cmap='grey')

In [ ]:
(singacc4_t[:1] != 0).sum()

In [ ]:
f5,a5 = plt.subplots(2)
a5[0].imshow(singacc4_image_abs[0], cmap='gray')
a5[1].imshow(singacc4_image_abs[1], cmap='gray')

In [ ]:
singacc4_t.shape

In [ ]:
import torch
semi_decent_res = torch.load('log/cmr_longtest/vis_imspace-cmr_longtest-lr1e-5-rlr1e-3-negJ0-grd0-lap0-pgr0-hel0-rec1e-9-sim1-dep3-dim64-tp2-e500-euler-ts0.1-siren-findif-aggrid-grid-rtol1e-6-atol1e-8-no-nrep-st0-seed0-db/res.pt', weights_only=False, map_location='cpu')

In [ ]:
semi_decent_res.keys()

In [ ]:
dec_moved_imgs = semi_decent_res['moved_imgs']
dec_moved_imgs = fastmri.complex_abs(dec_moved_imgs.squeeze().movedim(0,-1))
dec_moved_imgs.shape

In [ ]:
dec_moved_imgs.max(), dec_moved_imgs.min()

In [ ]:
f, a = plt.subplots(4)
t = 1
shape = dec_moved_imgs[t].shape
mask = singacc4_abs[t] != 0
gt = torch.log(singacc4_abs[t][mask].reshape(shape)+1e-9)
pred = torch.log(dec_moved_imgs[t]+1e-9)
a[0].imshow(gt, cmap="gray")
a[1].imshow(pred, cmap="gray")
a[2].imshow((gt-pred).abs(), cmap="gray")
a[3].imshow((dec_moved_imgs[t]-singacc4_abs[t][mask].reshape(shape)).abs(), cmap="gray", vmax=0.0125)

In [ ]:
from torchvision.utils import make_grid
imgs = singacc4_abs[t][mask].reshape(shape)
moved = dec_moved_imgs[t]

reg_last = make_grid([torch.stack([torch.log(imgs+1e-9), torch.log(moved+1e-9), torch.log(moved+1e-9)])], nrow=2, normalize=True)
reg_last2 = make_grid([torch.stack([imgs, moved, moved])], nrow=2, normalize=True)
reg_all = make_grid([torch.stack([im, torch.zeros_like(im), m_im]) for im, m_im in zip(imgs, moved)], nrow=5, normalize=True)

In [ ]:
f3, a3 = plt.subplots(2)
a3[0].imshow(reg_last.permute(1,2,0))
a3[1].imshow(reg_last2.permute(1,2,0))

In [ ]:
from src.vis_utils import create_animation_general, save_animation
im_html, _ = create_animation_general(sing_coil_image_abs, cmap="gray")

In [ ]:
im_html

In [ ]:
slice_kspace2 = T.to_tensor(slice_kspace)              # Convert from numpy array to pytorch tensor
slice_image = fastmri.ifft2c(slice_kspace2)           # Apply Inverse Fourier Transform to get the complex image
slice_image_abs = fastmri.complex_abs(slice_image)   # Compute absolute value to get a real image

In [ ]:
show_coils(slice_image_abs, [0, 3, 6], cmap='gray', vmax = 0.0005)

As we can see, each coil in a multi-coil MRI scan focusses on a different region of the image. These coils can be combined into the full image using the Root-Sum-of-Squares (RSS) transform.

In [ ]:
slice_image_rss = fastmri.rss(slice_image_abs, dim=0)

In [ ]:
plt.imshow(np.abs(slice_image_rss.numpy()), cmap='gray', vmax = 0.0015)

So far, we have been looking at fully-sampled multi-coil data. We can also apply the similar operator to the undersampled single-coil data to see how it can be changed to image domain.

In [ ]:
file_name = 'data/raw/cmrxrecon/ChallengeData/SingleCoil/Cine/TrainingSet/AccFactor04/P001/cine_sax.mat'
hf_s = h5py.File(file_name)

In [ ]:
print('Keys:', list(hf_s.keys()))

In [ ]:
# here to load the kspace data, the keys are differnt for different acceleration factors
newvalue_s = hf_s['kspace_single_sub04']
undersinglesample = newvalue_s["real"] + 1j*newvalue_s["imag"]
underslice = undersinglesample[0,5] # Choosing the middle slice of this volume

In [ ]:
underslice.shape

In [ ]:
plt.imshow(np.abs(underslice), cmap='gray', vmax = 0.001)

Let's see what the subsampled image looks like:

In [ ]:
underslice2 = T.to_tensor(underslice)              # Convert from numpy array to pytorch tensor
sampled_image = fastmri.ifft2c(underslice2)           # Apply Inverse Fourier Transform to get the complex image
sampled_image_abs = fastmri.complex_abs(sampled_image)   # Compute absolute value to get a real image
#sampled_image_rss = fastmri.rss(sampled_image_abs, dim=0)

In [ ]:
sampled_image_abs.shape

In [ ]:
plt.imshow(np.abs(sampled_image_abs.numpy()), cmap='gray',vmax = 0.0015)

Here we try to save the numpy to mat73 file, please use hdf5storage function. 
For a large numpy array, it may take a longer time (e.g. up to two minutes for a multi-coil file). 

In [ ]:
def savenumpy2mat(data, np_var, filepath):
    ''' 
    np_var: str, the name of the variable in the mat file.
    data: numpy, array to save.
    filepath: str, the path to save the mat file.
    '''
    savedict= {}
    savedict[np_var] = data
    hdf5storage.savemat(filepath, savedict)


In [ ]:
savenumpy2mat(underslice, 'underkspace', 'underkslice.mat')